# 노트북 12 — 양자 기초: 복소 벡터로서의 qubit

목표: $n$-qubit 상태가 *그저* 길이 $2^n$짜리 복소 벡터라는 사실에 대한 직관을 쌓고, Bell 상태가 실제로 동작하는 모습을 보는 것입니다. 사전 양자 지식은 필요하지 않습니다.

## bit에서 qubit으로

고전 bit은 0 또는 1입니다. **Qubit**은 $\mathbb{C}^2$ 위의 단위 벡터 $\alpha|0\rangle + \beta|1\rangle$이며, $|\alpha|^2 + |\beta|^2 = 1$을 만족합니다. $|\alpha|^2$과 $|\beta|^2$은 각각 측정 결과가 0 또는 1이 될 확률입니다.

$n$개의 qubit: 길이 $2^n$짜리 벡터입니다. 그게 전부입니다 — 모든 "이상함"은 복소수 위의 선형대수에 들어 있습니다.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pqc_edu.quantum.simulator import QuantumState, H, X, Z, CNOT_MATRIX, Rk

## 1-qubit 상태

$|0\rangle$을 만든 다음 Hadamard를 적용하여 균등한 중첩 상태로 만듭니다.

In [2]:
q = QuantumState(1)
print('before:', q.amps)
q.apply_1q(H, 0)
print('after H:', q.amps)
print('probability of 0:', abs(q.amps[0])**2)
print('probability of 1:', abs(q.amps[1])**2)

before: [1.+0.j 0.+0.j]
after H: [0.70710678+0.j 0.70710678+0.j]
probability of 0: 0.4999999999999999
probability of 1: 0.4999999999999999


## 측정은 상태를 붕괴(collapse)시킵니다

측정은 유일하게 유니타리가 아닌 단계입니다. $H$를 적용한 뒤 측정을 하면 0과 1이 각각 50% 확률로 나옵니다.

In [3]:
rng = np.random.default_rng(0)
counts = [0, 0]
for _ in range(1000):
    q = QuantumState(1)
    q.apply_1q(H, 0)
    counts[q.measure([0], rng)] += 1
print('0 count:', counts[0], '   1 count:', counts[1])
print('roughly balanced:', abs(counts[0] - counts[1]) < 100)

0 count: 473    1 count: 527
roughly balanced: True


## 얽힘(Entanglement): Bell 상태

2개의 qubit이 있습니다. 첫 번째에 $H$를 적용한 다음, 첫 번째를 제어 qubit으로 하여 CNOT을 적용합니다. 결과는 $(|00\rangle + |11\rangle)/\sqrt{2}$이며, 최대로 얽힌 상태입니다. 한쪽 qubit을 측정하면 다른 쪽의 결과가 강제로 결정됩니다.

In [4]:
rng = np.random.default_rng(1)
outcomes = []
for _ in range(500):
    q = QuantumState(2)
    q.apply_1q(H, 0)
    q.apply_controlled_1q(X, ctrl=0, target=1)
    outcomes.append(q.measure([0, 1], rng))
print('counts:', {i: outcomes.count(i) for i in range(4)})
print('only |00> (0) and |11> (3) appear:', set(outcomes) <= {0, 3})

counts: {0: 259, 1: 0, 2: 0, 3: 241}
only |00> (0) and |11> (3) appear: True


## 다음은

방금 우리는 물리학에서 가장 직관에 반하는 결과 중 하나를 약 30줄의 numpy로 재현하는 시뮬레이터를 만들었습니다. 다음 노트북에서는 QFT — Shor 알고리즘이 주기를 복원하게 해 주는 핵심 재료 — 를 만듭니다.

→ `13_qft_and_period.ipynb`